In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from scipy import stats
import warnings

In [ ]:
warnings.filterwarnings('ignore')

# --- CONFIGURATION ---
FILE_PATH = 'LSEG YEARLY EXTRA FEATURES.csv'
SEED = 42
CALIPER_VALUE = 0.2
STRATA_BINS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
TARGET = 'Has Controversy'

# Continuous variables
FEATURES_CONT = ['Company Market Cap', 'Return on Equity - Actual', 
                 'Independent Board Members', 'Total Debt Percentage of Total Equity']

# Extra features to improve PSM balance test
FEATURES_EXTRA = ['CAPxLEV', 'ROExLEV', 'CAPxROE', 'ROEsq', 'BOARDsq', 'CAPsq', 'CAPxBOARD']
# --------------------

# 1. Load data
df_raw = pd.read_csv(FILE_PATH)

# 2. Dummies for industry subsectors
# This step creates columns like 'TRBC Industry Name_Energy', etc.
df_mod = pd.get_dummies(df_raw, columns=['TRBC Industry Name'], drop_first=True)

# Identify all sector columns
sector_cols = [c for c in df_mod.columns if 'TRBC Industry Name_' in c]
X_cols = FEATURES_CONT + FEATURES_EXTRA + sector_cols

# Scale and calculate propensity scores
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_mod[X_cols])

ps_model = LogisticRegression(random_state=SEED, max_iter=1000)
ps_model.fit(X_scaled, df_mod[TARGET])
df_mod['PS'] = ps_model.predict_proba(X_scaled)[:, 1]

# 4. Matching (Without Replacement)
matched = []
for yr in df_mod['Year'].unique():
    yr_df = df_mod[df_mod['Year'] == yr]
    tr = yr_df[yr_df[TARGET] == 1]
    co = yr_df[yr_df[TARGET] == 0].copy()
    
    for _, t_row in tr.iterrows():
        if not co.empty:
            diff = (co['PS'] - t_row['PS']).abs()
            best_idx = diff.idxmin()
            if diff.loc[best_idx] <= CALIPER_VALUE:
                matched.extend([t_row, co.loc[best_idx]])
                co = co.drop(best_idx)

# 5. Balance tests
if matched:
    df_matched = pd.DataFrame(matched)
    df_matched['stratum'] = pd.cut(df_matched['PS'], bins=STRATA_BINS, 
                                   labels=[f"{STRATA_BINS[i]}-{STRATA_BINS[i+1]}" for i in range(len(STRATA_BINS)-1)])

    full_balance_results = []
    # Tests continuous variables as well as sector dummies
    vars_to_test = FEATURES_CONT + sector_cols
    
    for label in sorted(df_matched['stratum'].unique()):
        stratum_df = df_matched[df_matched['stratum'] == label]
        if stratum_df.empty or len(stratum_df[TARGET].unique()) < 2:
            continue
            
        for var in vars_to_test:
            t_grp = stratum_df[stratum_df[TARGET] == 1][var].dropna()
            c_grp = stratum_df[stratum_df[TARGET] == 0][var].dropna()
            
            if len(t_grp) > 1 and len(c_grp) > 1:
                m1, m2 = t_grp.mean(), c_grp.mean()
                v1, v2 = t_grp.var(), c_grp.var()
                
                # T-test
                _, p_val = stats.ttest_ind(t_grp, c_grp)
                
                # SMD calculation (Standardized Mean Difference)
                pooled_std = np.sqrt((v1 + v2) / 2)
                smd = abs((m1 - m2) / pooled_std) if pooled_std != 0 else 0
                
                full_balance_results.append({
                    'Stratum': label,
                    'Type': 'Sector' if var in sector_cols else 'Continu',
                    'Variabele': var.replace('TRBC Industry Name_', ''),
                    'Mean_Treated': m1,
                    'Mean_Control': m2,
                    'p_val': round(p_val, 4),
                    'SMD': round(smd, 4)
                })

    # Export to CSV
    df_output = pd.DataFrame(full_balance_results)
    df_output.to_csv('PSM_STRATA_BALANCE', index=False)


-> Analyse voltooid. Bestand 'PSM_FULL_STRATA_BALANCE_INCL_SECTORS.csv' aangemaakt.
